# Refit CYAIPE64NEW v?i m?u m?i v? replay d? li?u c?

Notebook n?y kh?i ph?c ph?n LightGBM h?p l? trong `CYAIPE64NEW.mdl`, th?m c?c m?u s?ch m?i trong `D:\Retrain_clean` v? d?ng m?t replay set c?n b?ng t? `Clean64`/`Virus64` ?? model ti?p t?c gi? ki?n th?c c?a c? hai l?p.

- Model g?c kh?ng b? ghi ??.
- Refit gi? nguy?n 100 c?y v? ch? c?p nh?t gi? tr? l?.
- `DECAY_RATE = 0.80` gi? 80% gi? tr? l? c? v? nh?n 20% gi? tr? h?c t? d? li?u refit c?n b?ng.
- M?u replay validation kh?ng tham gia refit. Notebook kh?ng l?u model n?u accuracy c? gi?m qu? 1%, false positive c? t?ng, ho?c m?u m?i kh?ng ???c c?i thi?n.
- Model ??u ra ???c l?u b?ng `Booster.save_model()` v? ???c reload ngay ?? ki?m tra ??nh d?ng.

In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "1"

import collections
import re
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np


# Ch?y Jupyter t? th? m?c ch?a notebook n?y.
PROJECT_DIR = Path.cwd()
MODEL_PATH = PROJECT_DIR / "CYAIPE64NEW.mdl"
OUTPUT_PATH = PROJECT_DIR / "CYAIPE64NEW_refit.mdl"

# M?u m?i c?n s?a false positive.
NEW_CLEAN_DIR = Path(r"D:\Retrain_clean")

# Replay d? li?u c? ?? refit c? ?? c? nh?n 0 v? 1.
REPLAY_CLEAN_DIR = Path(r"F:\Data_Train_PE\Data_Train\Clean64")
REPLAY_MALWARE_DIR = Path(r"F:\Data_Train_PE\Data_Train\Virus64")
REPLAY_TRAIN_PER_CLASS = 250
REPLAY_VALID_PER_CLASS = 150
RANDOM_SEED = 42

EXPECTED_FEATURES = 1661
NEW_SAMPLE_WEIGHT = 1.0
DECAY_RATE = 0.80
MAX_VALIDATION_ACCURACY_DROP = 0.01

if not 0.0 < DECAY_RATE < 1.0:
    raise ValueError("DECAY_RATE ph?i n?m trong kho?ng (0, 1)")
if REPLAY_TRAIN_PER_CLASS <= 0 or REPLAY_VALID_PER_CLASS <= 0:
    raise ValueError("S? l??ng replay ph?i l?n h?n 0")
if NEW_SAMPLE_WEIGHT <= 0:
    raise ValueError("NEW_SAMPLE_WEIGHT ph?i l?n h?n 0")

print("LightGBM:", lgb.__version__)
print("Model g?c:", MODEL_PATH.resolve())
print("Model ??u ra:", OUTPUT_PATH.resolve())

In [ ]:
def load_legacy_booster(model_path):
    """Load pickle chu?n ho?c kh?i ph?c LightGBM text b? d?nh ?u?i pickle."""
    model_path = Path(model_path)
    if not model_path.is_file():
        raise FileNotFoundError(
            f"Kh?ng t?m th?y {model_path}. "
            "H?y ch?y Jupyter t? th? m?c Data_Preprocessing."
        )

    raw = model_path.read_bytes()

    # Pickle chu?n do pickle.dump(Booster, file) t?o ra.
    if raw.startswith(b"\x80"):
        import pickle

        # Ch? unpickle model do ch?nh pipeline tin c?y n?y t?o ra.
        with model_path.open("rb") as file:
            booster = pickle.load(file)
        if hasattr(booster, "booster_"):
            booster = booster.booster_
        if not isinstance(booster, lgb.Booster):
            raise TypeError(f"??i t??ng trong {model_path} kh?ng ph?i LightGBM Booster")
        return booster, {
            "format": "pickle",
            "file_bytes": len(raw),
            "model_bytes": len(raw),
            "ignored_bytes": 0,
        }

    # CYAIPE64NEW.mdl hi?n t?i b?t ??u b?ng model text nh?ng c?n ?u?i pickle.
    if raw.startswith(b"tree\n"):
        marker = b"\npandas_categorical:"
        marker_position = raw.find(marker)
        if marker_position < 0:
            raise ValueError("Kh?ng t?m th?y d?ng pandas_categorical trong model")

        line_end = raw.find(b"\n", marker_position + len(marker))
        model_end = len(raw) if line_end < 0 else line_end + 1
        model_bytes = raw[:model_end]
        ignored_bytes = len(raw) - model_end

        try:
            model_text = model_bytes.decode("utf-8")
        except UnicodeDecodeError as exc:
            raise ValueError("Ph?n LightGBM text kh?ng ph?i UTF-8 h?p l?") from exc

        booster = lgb.Booster(model_str=model_text)
        return booster, {
            "format": "recovered LightGBM text",
            "file_bytes": len(raw),
            "model_bytes": len(model_bytes),
            "ignored_bytes": ignored_bytes,
        }

    raise ValueError(
        f"Kh?ng nh?n di?n ???c ??nh d?ng {model_path}; "
        f"byte ??u: {raw[:16].hex(' ')}"
    )

In [ ]:
print("1. ?ang t?i model c?...")
start_time = time.perf_counter()
old_model, load_info = load_legacy_booster(MODEL_PATH)

if old_model.num_feature() != EXPECTED_FEATURES:
    raise ValueError(
        f"Model c? {old_model.num_feature()} features, "
        f"nh?ng pipeline c?n {EXPECTED_FEATURES}"
    )
if old_model.num_trees() <= 0:
    raise ValueError("Model kh?ng c? c?y n?o")

print(f"-> Load th?nh c?ng sau {time.perf_counter() - start_time:.2f} gi?y")
print("-> ??nh d?ng:", load_info["format"])
print(f"-> K?ch th??c file: {load_info['file_bytes']:,} byte")
print(f"-> Ph?n model h?p l?: {load_info['model_bytes']:,} byte")
print(f"-> B? qua ph?n ?u?i h?ng: {load_info['ignored_bytes']:,} byte")
print("-> S? c?y:", old_model.num_trees())
print("-> S? features:", old_model.num_feature())

In [ ]:
# Pipeline gi?ng Data_Processing1.ipynb:
# 775 numeric + 10 section + 863 import token + 13 handcrafted = 1661.
section_path = PROJECT_DIR / "PE64Section"
import_path = PROJECT_DIR / "PE64D"

sections = section_path.read_text(encoding="utf-8", errors="ignore").split()
section2id = {name.lower(): index for index, name in enumerate(sections, 1)}
section2id["null"] = 0

imports = [
    line.strip()
    for line in import_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    if line.strip()
]
token2id = {name: index for index, name in enumerate(imports)}

HAND_KEYS = [
    "n_imports_total", "n_dlls_total", "n_ordinals_total",
    "cnt_crt_imports", "ratio_crt", "cnt_dynload", "cnt_mem",
    "cnt_proc_enum", "cnt_reg", "has_net_dll", "has_crypto_dll",
    "has_dynload_and_mem", "has_proc_enum_and_mem",
]

if len(token2id) != len(imports):
    raise ValueError("PE64D ch?a token tr?ng l?p; th? t? feature kh?ng c?n ??ng tin c?y")

calculated_features = 775 + 10 + len(token2id) + len(HAND_KEYS)
if calculated_features != EXPECTED_FEATURES:
    raise ValueError(
        f"Pipeline t?o {calculated_features} features thay v? {EXPECTED_FEATURES}"
    )

CRT = re.compile(r"^(api-ms-win-crt-|msvcp|vcruntime|ucrtbase)", re.I)
DECORATION = re.compile(r"@\d+$")
WINAPI = {
    "kernel32.dll", "ntdll.dll", "advapi32.dll", "user32.dll",
    "gdi32.dll", "shell32.dll", "shlwapi.dll", "ole32.dll",
    "oleaut32.dll", "comctl32.dll", "wininet.dll", "winhttp.dll",
    "ws2_32.dll", "urlmon.dll", "crypt32.dll", "bcrypt.dll",
    "secur32.dll", "iphlpapi.dll", "psapi.dll", "dbghelp.dll",
    "wtasapi32.dll",
}
DYN = {
    "loadlibrarya", "loadlibraryw", "loadlibraryexw",
    "loadlibraryexa", "getprocaddress",
}
MEM = {"virtualalloc", "virtualprotect", "virtualquery", "virtualfree"}
PROC = {
    "createtoolhelp32snapshot", "process32firstw", "process32nextw",
    "thread32first", "thread32next",
}
REG = {
    "regopenkeyexw", "regsetvalueexw", "regcreatekeyexw",
    "regqueryvalueexw", "regdeletevaluew",
}
NET = {"wininet.dll", "winhttp.dll", "ws2_32.dll", "urlmon.dll"}
CRYPTO = {"crypt32.dll", "bcrypt.dll"}


def make_import_vector(raw_tokens):
    tokens = set()
    features = collections.Counter()
    dlls = set()
    total = 0
    ordinal = 0
    crt_count = 0

    for raw in raw_tokens:
        position = raw.lower().find(".dll")
        if position < 0:
            continue

        dll = raw[:position + 4].lower()
        function = raw[position + 4:].strip()
        dlls.add(dll)

        if not CRT.match(dll):
            tokens.add(f"dll:{dll}")
        if not function:
            continue

        total += 1
        if CRT.match(dll):
            crt_count += 1
            continue

        function = (
            f"ord_{function}"
            if function.isdigit()
            else DECORATION.sub("", function).lower()
        )
        ordinal += int(function.startswith("ord_"))

        if dll in WINAPI:
            tokens.add(f"{dll}!{function}")
            features["cnt_dynload"] += int(function in DYN)
            features["cnt_mem"] += int(function in MEM)
            features["cnt_proc_enum"] += int(function in PROC)
            features["cnt_reg"] += int(function in REG)

        features["has_net_dll"] |= int(dll in NET)
        features["has_crypto_dll"] |= int(dll in CRYPTO)

    features.update(
        n_imports_total=total,
        n_dlls_total=len(dlls),
        n_ordinals_total=ordinal,
        cnt_crt_imports=crt_count,
        ratio_crt=crt_count / total if total else 0.0,
    )
    features["has_dynload_and_mem"] = int(
        features["cnt_dynload"] > 0 and features["cnt_mem"] > 0
    )
    features["has_proc_enum_and_mem"] = int(
        features["cnt_proc_enum"] > 0 and features["cnt_mem"] > 0
    )

    vector = np.zeros(len(token2id) + len(HAND_KEYS), dtype=np.float32)
    for token in tokens:
        token_index = token2id.get(token)
        if token_index is not None:
            vector[token_index] = 1.0
    for index, key in enumerate(HAND_KEYS, len(token2id)):
        vector[index] = features[key]
    return vector


def make_feature(path):
    data = path.read_text(encoding="latin-1").strip().split("\t")

    # T??ng th?ch c?c feature c? thi?u block 256 gi? tr? t?i v? tr? 292.
    if len(data) < 776:
        data = data[:292] + ["0.0"] * 256 + data[292:]
    if len(data) < 777:
        raise ValueError(f"{path.name}: sai ??nh d?ng feature")

    numeric = np.asarray(data[:775], dtype=np.float32)
    names = data[775].lower().split()[:10]
    names += ["null"] * (10 - len(names))
    section_vector = np.asarray(
        [section2id.get(name, 0) for name in names],
        dtype=np.float32,
    )

    raw_imports = re.sub(r"[\n\t]", "", "".join(data[776:])).lower().split()
    feature = np.concatenate(
        [numeric, section_vector, make_import_vector(raw_imports)]
    )

    if len(feature) != EXPECTED_FEATURES:
        raise ValueError(
            f"{path.name}: {len(feature)} features thay v? {EXPECTED_FEATURES}"
        )
    return feature


print("Pipeline features h?p l?:", calculated_features)

In [ ]:
def load_feature_split(directory, train_count, valid_count, rng):
    """L?y hai t?p file kh?ng tr?ng nhau v? b? qua file feature l?i."""
    directory = Path(directory)
    if not directory.is_dir():
        raise FileNotFoundError(f"Kh?ng t?m th?y th? m?c replay: {directory}")

    paths = np.asarray(
        sorted(path for path in directory.iterdir() if path.is_file()),
        dtype=object,
    )
    if len(paths) < train_count + valid_count:
        raise ValueError(
            f"{directory} ch? c? {len(paths)} files; "
            f"c?n ?t nh?t {train_count + valid_count}"
        )

    train_features = []
    train_paths = []
    valid_features = []
    valid_paths = []
    failed_paths = []

    for index in rng.permutation(len(paths)):
        file_path = paths[index]
        try:
            feature = make_feature(file_path)
        except (OSError, UnicodeError, ValueError) as exc:
            failed_paths.append((file_path, str(exc)))
            continue

        if len(train_features) < train_count:
            train_features.append(feature)
            train_paths.append(file_path)
        elif len(valid_features) < valid_count:
            valid_features.append(feature)
            valid_paths.append(file_path)
        else:
            break

    if len(train_features) != train_count or len(valid_features) != valid_count:
        raise ValueError(
            f"{directory}: kh?ng ?? feature h?p l? sau khi b? file l?i; "
            f"train={len(train_features)}, validation={len(valid_features)}"
        )

    return (
        np.ascontiguousarray(np.vstack(train_features), dtype=np.float32),
        train_paths,
        np.ascontiguousarray(np.vstack(valid_features), dtype=np.float32),
        valid_paths,
        failed_paths,
    )


print("2. ?ang ??c m?u s?ch m?i...")
if not NEW_CLEAN_DIR.is_dir():
    raise FileNotFoundError(f"Kh?ng t?m th?y th? m?c: {NEW_CLEAN_DIR}")

new_clean_files = sorted(path for path in NEW_CLEAN_DIR.iterdir() if path.is_file())
if not new_clean_files:
    raise ValueError(f"Kh?ng c? file n?o trong {NEW_CLEAN_DIR}")

X_new = np.ascontiguousarray(
    np.vstack([make_feature(path) for path in new_clean_files]),
    dtype=np.float32,
)
y_new = np.zeros(len(new_clean_files), dtype=np.int32)

print("3. ?ang t?o replay train/validation c?n b?ng...")
rng = np.random.default_rng(RANDOM_SEED)
(
    X_old_clean_train,
    old_clean_train_files,
    X_old_clean_valid,
    old_clean_valid_files,
    clean_failures,
) = load_feature_split(
    REPLAY_CLEAN_DIR,
    REPLAY_TRAIN_PER_CLASS,
    REPLAY_VALID_PER_CLASS,
    rng,
)
(
    X_old_malware_train,
    old_malware_train_files,
    X_old_malware_valid,
    old_malware_valid_files,
    malware_failures,
) = load_feature_split(
    REPLAY_MALWARE_DIR,
    REPLAY_TRAIN_PER_CLASS,
    REPLAY_VALID_PER_CLASS,
    rng,
)

# Refit: m?u m?i + replay s?ch c? + replay malware c?.
X_refit = np.ascontiguousarray(
    np.vstack([X_new, X_old_clean_train, X_old_malware_train]),
    dtype=np.float32,
)
y_refit = np.concatenate([
    y_new,
    np.zeros(len(X_old_clean_train), dtype=np.int32),
    np.ones(len(X_old_malware_train), dtype=np.int32),
])
refit_weights = np.ones(len(y_refit), dtype=np.float32)
refit_weights[:len(X_new)] = NEW_SAMPLE_WEIGHT

# Validation c? kh?ng tham gia refit.
X_old_valid = np.ascontiguousarray(
    np.vstack([X_old_clean_valid, X_old_malware_valid]),
    dtype=np.float32,
)
y_old_valid = np.concatenate([
    np.zeros(len(X_old_clean_valid), dtype=np.int32),
    np.ones(len(X_old_malware_valid), dtype=np.int32),
])

for name, matrix in {
    "X_new": X_new,
    "X_refit": X_refit,
    "X_old_valid": X_old_valid,
}.items():
    if matrix.ndim != 2 or matrix.shape[1] != old_model.num_feature():
        raise ValueError(f"{name} c? shape kh?ng h?p l?: {matrix.shape}")
    if not np.isfinite(matrix).all():
        raise ValueError(f"{name} ch?a NaN ho?c Infinity")

if set(np.unique(y_refit)) != {0, 1}:
    raise ValueError("T?p refit ph?i c? ?? c? nh?n 0 v? 1")

print("-> Mau sach moi:", len(X_new))
print("-> Replay train:", len(X_old_clean_train), "s?ch +", len(X_old_malware_train), "malware")
print("-> Replay validation:", len(X_old_clean_valid), "s?ch +", len(X_old_malware_valid), "malware")
print("-> X_refit:", X_refit.shape, "| labels:", np.bincount(y_refit))
print("-> File replay l?i b? b? qua:", len(clean_failures) + len(malware_failures))

In [ ]:
def binary_metrics(model, data, labels):
    scores = model.predict(data)
    predictions = scores >= 0.5
    labels = np.asarray(labels, dtype=np.int32)
    clean_mask = labels == 0
    malware_mask = labels == 1
    return {
        "scores": scores,
        "accuracy": float(np.mean(predictions == labels)),
        "false_positive": int(np.sum(predictions[clean_mask])),
        "false_negative": int(np.sum(~predictions[malware_mask])),
    }


print("4. ??nh gi? model c?...")
before_new = binary_metrics(old_model, X_new, y_new)
before_old = binary_metrics(old_model, X_old_valid, y_old_valid)

print(f"5. ?ang refit v?i DECAY_RATE={DECAY_RATE:.2f}...")
start_time = time.perf_counter()
new_model = old_model.refit(
    data=X_refit,
    label=y_refit,
    weight=refit_weights,
    decay_rate=DECAY_RATE,
)
elapsed = time.perf_counter() - start_time

if new_model.num_trees() != old_model.num_trees():
    raise RuntimeError("Refit ?? l?m thay ??i s? c?y ngo?i d? ki?n")
if new_model.num_feature() != old_model.num_feature():
    raise RuntimeError("Refit ?? l?m thay ??i s? features ngo?i d? ki?n")

after_new = binary_metrics(new_model, X_new, y_new)
after_old = binary_metrics(new_model, X_old_valid, y_old_valid)
accuracy_drop = before_old["accuracy"] - after_old["accuracy"]

print(f"-> Refit ho?n t?t sau {elapsed:.2f} gi?y")
print("\nK?t qu? t?ng m?u m?i:")
for file_path, old_score, new_score in zip(
    new_clean_files,
    before_new["scores"],
    after_new["scores"],
):
    print(
        f"{file_path.name} | "
        f"before={old_score:.6f} | "
        f"after={new_score:.6f} | "
        f"delta={new_score - old_score:+.6f}"
    )

print("\nT?ng k?t m?u s?ch m?i:")
print(
    f"-> ?i?m trung b?nh: "
    f"{before_new['scores'].mean():.6f} -> {after_new['scores'].mean():.6f}"
)
print(
    f"-> False positive: "
    f"{before_new['false_positive']} -> {after_new['false_positive']}"
)

print("\nT?ng k?t replay validation c? (kh?ng tham gia refit):")
print(
    f"-> Accuracy: {before_old['accuracy']:.4%} -> {after_old['accuracy']:.4%} "
    f"(drop={accuracy_drop:.4%})"
)
print(
    f"-> False positive c?: "
    f"{before_old['false_positive']} -> {after_old['false_positive']}"
)
print(
    f"-> False negative c?: "
    f"{before_old['false_negative']} -> {after_old['false_negative']}"
)

# Quality gates: ch? l?u khi m?u m?i t?t h?n v? ki?n th?c c? ???c gi? ?? t?t.
if after_new["false_positive"] >= before_new["false_positive"]:
    raise RuntimeError("Refit kh?ng gi?m false positive tr?n m?u m?i; model kh?ng ???c l?u")
if accuracy_drop > MAX_VALIDATION_ACCURACY_DROP + 1e-12:
    raise RuntimeError(
        f"Accuracy c? gi?m {accuracy_drop:.4%}, v??t ng??ng "
        f"{MAX_VALIDATION_ACCURACY_DROP:.4%}; model kh?ng ???c l?u"
    )
if after_old["false_positive"] > before_old["false_positive"]:
    raise RuntimeError("False positive tr?n replay c? t?ng; model kh?ng ???c l?u")

print("\n6. ?ang l?u model refit ? ??nh d?ng LightGBM native...")
new_model.save_model(str(OUTPUT_PATH))

# Reload ngay ?? ch?c ch?n file ??u ra kh?ng c?n l?i pickle/text nh? model c?.
verified_model = lgb.Booster(model_file=str(OUTPUT_PATH))
verified_new = verified_model.predict(X_new)
verified_old = verified_model.predict(X_old_valid)

if verified_model.num_trees() != new_model.num_trees():
    raise RuntimeError("Model reload c? s? c?y kh?ng kh?p")
if verified_model.num_feature() != EXPECTED_FEATURES:
    raise RuntimeError("Model reload c? s? features kh?ng kh?p")
if not np.allclose(after_new["scores"], verified_new, rtol=1e-10, atol=1e-12):
    raise RuntimeError("D? ?o?n m?u m?i thay ??i sau khi reload model")
if not np.allclose(after_old["scores"], verified_old, rtol=1e-10, atol=1e-12):
    raise RuntimeError("D? ?o?n validation thay ??i sau khi reload model")

print("-> ?? l?u v? reload th?nh c?ng:", OUTPUT_PATH.resolve())
print("-> S? c?y:", verified_model.num_trees())
print("-> S? features:", verified_model.num_feature())